In [2]:
import pandas as pd

df = pd.read_csv(r"W:\Car_Hacking_Challenge_Dataset_rev20Mar2021\0_Preliminary\0_Training\Pre_train_D_0.csv")

print(df.shape)
print(df.dtypes)
print(df.head(20))
print(df["Class"].value_counts())
print(df["Arbitration_ID"].nunique())

(179346, 5)
Timestamp         float64
Arbitration_ID        str
DLC                 int64
Data                  str
Class                 str
dtype: object
       Timestamp Arbitration_ID  DLC                     Data   Class
0   1.597708e+09            260    8  06 25 05 30 FF CF 71 55  Normal
1   1.597708e+09            329    8  4A C5 7E 8C 31 2D 01 10  Normal
2   1.597708e+09            38D    8  00 00 49 00 90 7F FE 01  Normal
3   1.597708e+09            420    8  50 1E 00 C8 FC 4F 6A 00  Normal
4   1.597708e+09            421    8  FE 07 00 FF E3 7F 00 52  Normal
5   1.597708e+09            153    8  20 80 10 FF 00 FF A0 4E  Normal
6   1.597708e+09            220    8  12 24 7C 00 00 6D 10 F6  Normal
7   1.597708e+09            340    8  00 00 00 24 36 01 25 30  Normal
8   1.597708e+09            389    8  00 00 00 20 00 00 C2 00  Normal
9   1.597708e+09            47F    8  00 EC FF FA 00 78 00 36  Normal
10  1.597708e+09            130    8  E4 80 AC 80 00 00 07 95  Normal
11  

In [3]:
duration = df["Timestamp"].max() - df["Timestamp"].min()
print(duration)
print(len(df)/duration)

74.65505194664001
2402.329049723095


Dataset overview: 75 seconds, 179,346 messages, about 2,400 per second, 73 IDs.

In [4]:
df["Arbitration_ID"].value_counts()

Arbitration_ID
260    7466
153    7466
220    7466
329    7465
340    7465
       ... 
5B0      75
043      75
5BE      74
07F      74
57F      37
Name: count, Length: 73, dtype: int64

In [12]:
one_id = df[df["Arbitration_ID"] == "260"]
print(len(one_id))
print(one_id["Timestamp"].head(10))

7466
0      1.597708e+09
27     1.597708e+09
46     1.597708e+09
72     1.597708e+09
93     1.597708e+09
121    1.597708e+09
148    1.597708e+09
175    1.597708e+09
194    1.597708e+09
224    1.597708e+09
Name: Timestamp, dtype: float64


In [13]:
t = one_id["Timestamp"] - df["Timestamp"].min()
print(t.head(10))

0      0.000000
27     0.011067
46     0.020065
72     0.030849
93     0.041443
121    0.050866
148    0.060842
175    0.071488
194    0.080039
224    0.091151
Name: Timestamp, dtype: float64


In [15]:
gaps = one_id["Timestamp"].diff()
print(gaps.head(10))
print(gaps.mean())
print(gaps.std())

0           NaN
27     0.011067
46     0.008998
72     0.010784
93     0.010594
121    0.009423
148    0.009976
175    0.010646
194    0.008551
224    0.011112
Name: Timestamp, dtype: float64
0.01000057787042453
0.0008062583280069837


ID 260 timing. ID 260 appears 7,466 times in about 75 seconds, roughly 100 times per second. The gap between consecutive 260 messages has a mean of 10.0006 ms and a standard deviation of 0.8 ms. So the ECU sends this ID on a fixed 10 ms clock, and the small wobble comes from messages sometimes waiting for the bus to be free.

In [18]:
import glob, os

base = r"W:\Car_Hacking_Challenge_Dataset_rev20Mar2021\0_Preliminary\0_Training"

for path in sorted(glob.glob(os.path.join(base, "*.csv"))):
    d = pd.read_csv(path)
    print(os.path.basename(path), d.shape, dict(d["Class"].value_counts()))

Pre_train_D_0.csv (179346, 5) {'Normal': np.int64(179346)}
Pre_train_D_1.csv (806390, 6) {'Normal': np.int64(733752), 'Attack': np.int64(72638)}
Pre_train_D_2.csv (889395, 6) {'Normal': np.int64(811532), 'Attack': np.int64(77863)}
Pre_train_S_0.csv (180686, 5) {'Normal': np.int64(180686)}
Pre_train_S_1.csv (799292, 6) {'Normal': np.int64(727749), 'Attack': np.int64(71543)}
Pre_train_S_2.csv (817042, 6) {'Normal': np.int64(739678), 'Attack': np.int64(77364)}


In [19]:
attack_df = pd.read_csv(r"W:\Car_Hacking_Challenge_Dataset_rev20Mar2021\0_Preliminary\0_Training\Pre_train_D_1.csv")

print(attack_df.shape)
print(attack_df["SubClass"].value_counts())

(806390, 6)
SubClass
Normal      733752
Flooding     38521
Fuzzing      22620
Replay       10509
Spoofing       988
Name: count, dtype: int64


Attack file contents. Pre_train_D_1.csv has 806,390 messages, of which 733,752 are normal and 72,638 are labelled as attacks. There are four attack types: Flooding (38,521 messages), Fuzzing (22,620), Replay (10,509) and Spoofing (988). Spoofing is much smaller than the others, with 494 messages each on IDs 553 and 366.

In [20]:
flood = attack_df[attack_df["SubClass"] == "Flooding"]
print(flood["Arbitration_ID"].value_counts().head(10))

Arbitration_ID
000    38521
Name: count, dtype: int64


In [21]:
print("000" in df["Arbitration_ID"].values)

False


Flooding: uses ID 000, which never appears in clean traffic.

In [24]:
spoof = attack_df[attack_df["SubClass"] == "Spoofing"]
print(spoof["Arbitration_ID"].value_counts().head(10))

Arbitration_ID
553    494
366    494
Name: count, dtype: int64


In [25]:
print(df["Arbitration_ID"].unique())

<StringArray>
['260', '329', '38D', '420', '421', '153', '220', '340', '389', '47F', '130',
 '140', '251', '2B0', '164', '356', '366', '367', '368', '381', '387', '394',
 '453', '470', '490', '495', '500', '391', '4F1', '484', '386', '479', '436',
 '541', '485', '492', '4C9', '48A', '58B', '48C', '544', '553', '50C', '52A',
 '507', '5B0', '5CD', '44E', '520', '49F', '572', '483', '50A', '559', '410',
 '412', '50E', '563', '568', '42D', '53E', '4CB', '53B', '5A6', '4A9', '53F',
 '593', '043', '4A7', '4A2', '5BE', '07F', '57F']
Length: 73, dtype: str


Spoofing: uses legitimate IDs 553 and 366, so the unseen-ID check fails.

In [26]:
clean_553 = df[df["Arbitration_ID"] == "553"]["Timestamp"].diff()
print(clean_553.mean(), clean_553.std())

0.2000011720222698 0.0002993749920116


In [27]:
att_553 = attack_df[attack_df["Arbitration_ID"] == "553"].sort_values("Timestamp")["Timestamp"].diff()
print(att_553.describe())

count    2280.000000
mean        0.133598
std         0.076351
min         0.000217
25%         0.060160
50%         0.199692
75%         0.200012
max         0.344891
Name: Timestamp, dtype: float64


In [28]:
short = att_553 < 0.19
print(short.sum(), short.mean())

1044 0.45769399386234105


In [29]:
short_clean = clean_553 < 0.19
print(short_clean.sum(), short_clean.mean())

0 0.0


ID 553 timing check. In clean traffic, ID 553 arrives every 200 ms with a standard deviation of about 0.3 ms. In the spoofing attack file the mean gap drops to 134 ms, but the median stays at 199.7 ms, so the mean hides what is really happening. Counting gaps shorter than 190 ms works better: 1,044 of 2,280 gaps (46 percent) in the attack file, and 0 in clean traffic.